In [1]:
import os, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
print(sys.executable)

d:\GithubRepos\llm-internals\.venv\Scripts\python.exe


In [2]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-tr", split="test")
pairs = [(r["translation"]["en"], r["translation"]["tr"]) for r in ds]
print(len(pairs))
print(pairs[0])

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/139k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/60.1M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

2000
('Russian leaders reacted to the announcement of US plans to place missile defence bases in eastern Poland.', "Rus liderleri Amerika Birleşik Devletleri'nin Polonya'nın doğusuna füze üssü kurma planlarına tepkili.")


In [3]:
import tiktoken
from transformers import AutoTokenizer

o200k = tiktoken.get_encoding("o200k_base")
cl100k = tiktoken.get_encoding("cl100k_base")
tr_bert = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

def count(tok, text):
    if hasattr(tok, "encode_ordinary"):
        return len(tok.encode_ordinary(text))
    return len(tok.encode(text, add_special_tokens=False))

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

In [4]:
sample = "Evlerinizden çıkmayın."
for name, tok in [("o200k", o200k), ("cl100k", cl100k), ("bert-tr", tr_bert)]:
    print(name, count(tok, sample))

o200k 8
cl100k 10
bert-tr 7


In [5]:
import pandas as pd

en_texts = [en for en, tr in pairs]
tr_texts = [tr for en, tr in pairs]

def measure(tok, texts):
    tokens = words = chars = 0
    for t in texts:
        tokens += count(tok, t)
        words += len(t.split())
        chars += len(t)
    return {"sentences": len(texts), "words": words, "chars": chars, "tokens": tokens}

tokenizers = {"o200k_base": o200k, "cl100k_base": cl100k, "bert-tr": tr_bert}
langs = {"en": en_texts, "tr": tr_texts}

rows = []
for tname, tok in tokenizers.items():
    for lang, texts in langs.items():
        rows.append({"tokenizer": tname, "lang": lang, **measure(tok, texts)})

df = pd.DataFrame(rows)
df["tok/sent"] = df["tokens"] / df["sentences"]
df["tok/word"] = df["tokens"] / df["words"]
df["tok/char"] = df["tokens"] / df["chars"]
df.round(3)

,tokenizer,lang,sentences,words,chars,tokens,tok/sent,tok/word,tok/char
0,o200k_base,en,2000,15979,83053,20666,10.333,1.293,0.249
1,o200k_base,tr,2000,11249,77989,25146,12.573,2.235,0.322
2,cl100k_base,en,2000,15979,83053,21578,10.789,1.350,0.260
3,cl100k_base,tr,2000,11249,77989,32115,16.058,2.855,0.412
4,bert-tr,en,2000,15979,83053,33578,16.789,2.101,0.404
5,bert-tr,tr,2000,11249,77989,18814,9.407,1.673,0.241


In [6]:
pivot = df.pivot(index="tokenizer", columns="lang", values="tokens")
pivot["tr/en"] = pivot["tr"] / pivot["en"]
pivot.round(3)

lang,en,tr,tr/en
tokenizer,,,
bert-tr,33578,18814,0.560
cl100k_base,21578,32115,1.488
o200k_base,20666,25146,1.217


## Result

- On `o200k_base` (GPT-4o family), Turkish needs **1.22x** the tokens of English for the same content. On `cl100k_base` (GPT-4) the gap is 1.49x.
- The larger vocabulary helped Turkish disproportionately: moving from cl100k to o200k cut Turkish tokens by 22% and English tokens by only 4%.
- Tokens per word overstates the gap (1.73x) because Turkish packs more meaning into each word: the same content took 30% fewer words in Turkish.
- A Turkish-trained tokenizer (`dbmdz/bert-base-turkish-cased`) reverses the picture: 25% fewer Turkish tokens than o200k, 62% more English tokens.
- Practical effect: a 250-token chunk of Turkish text holds about 18% less content than a 250-token English chunk, and Turkish input costs about 22% more per request.

Corpus note: OPUS-100 en-tr test split, 2000 short conversational sentence pairs. Ratios for long formal documents may differ.